# 🍉 Watermelon Body 추출 모델 학습 (YOLOv8s-seg)

COCO 정답 데이터(`Watermelon.coco`)로 수박 **body 영역을 세그멘테이션**하는 모델을 직접 학습합니다.

설계 명세: `Watermelon.coco/train/extract.md`
- 5-Fold Cross-Validation
- Albumentations **오프라인 증강**(Hue 제외) — Train 셋만
- YOLOv8s-seg 파인튜닝, **내부 증강 OFF**
- 평가: Mask mAP50 / mAP50-95 의 5-Fold 평균

> ⚠️ 실데이터 반영: body는 `category_id=1`(이름 'body'로 조회), 무효 이미지(100·빈어노) 제외.

**실행 전:** 런타임 → 런타임 유형 변경 → **GPU(T4)** 선택

## Phase 0. 데이터 준비 (Google Drive 마운트 & 압축 해제)
`Watermelon.coco` 폴더를 zip으로 만들어 `MyDrive/watermelon/Watermelon.coco.zip` 에 올려두세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 업로드해 둔 zip 경로에 맞게 수정
!cp "/content/drive/MyDrive/watermelon/Watermelon.coco.zip" /content/
!unzip -q /content/Watermelon.coco.zip -d /content/data
!ls /content/data/Watermelon.coco/train | head

## Phase 1. 환경 설치

In [ ]:
!pip install -q ultralytics albumentations
import ultralytics; ultralytics.checks()

## Phase 2. COCO 로드 → 유효 이미지 추출 → 5-Fold 분할
- 이름 `'body'`로 카테고리 조회 (category_id=1)
- 빈/퇴화 폴리곤 제거 → `31.jpg` 빈 어노 방어
- body 있는 이미지만 사용 → `100.jpg` 자동 제외

In [ ]:
import json, os, cv2, numpy as np
from collections import defaultdict
from sklearn.model_selection import KFold

ROOT = "/content/data/Watermelon.coco/train"
COCO = f"{ROOT}/_annotations.coco.json"
coco = json.load(open(COCO, encoding="utf-8"))

body_id = next(c["id"] for c in coco["categories"] if c["name"] == "body")
print("body category_id =", body_id)

imgs = {im["id"]: im for im in coco["images"]}
body_by_img = defaultdict(list)
for a in coco["annotations"]:
    if a["category_id"] != body_id:                continue
    seg = a.get("segmentation")
    if not seg or isinstance(seg, dict):           continue   # 빈/RLE 제외
    if len(seg[0]) < 6:                            continue   # 점<3 제외
    body_by_img[a["image_id"]].append(a)

valid_ids = [i for i in imgs if body_by_img[i]]
print(f"유효 이미지: {len(valid_ids)} / {len(imgs)}")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(valid_ids))
for k, (tr, va) in enumerate(folds):
    print(f"  fold{k}: train {len(tr)} / val {len(va)}")

## Phase 3. 변환·증강 유틸 (COCO polygon → YOLO-seg txt)

In [ ]:
import albumentations as A

def write_yolo_label(ann_list, im, out_txt):
    W, H = im["width"], im["height"]; lines = []
    for a in ann_list:
        p = np.array(a["segmentation"][0], float).reshape(-1, 2)
        p[:, 0] /= W; p[:, 1] /= H
        lines.append("0 " + " ".join(f"{v:.6f}" for v in p.flatten()))  # class 0 = body
    open(out_txt, "w").write("\n".join(lines))

def poly_to_mask(ann_list, im):
    m = np.zeros((im["height"], im["width"]), np.uint8)
    for a in ann_list:
        cv2.fillPoly(m, [np.array(a["segmentation"][0], np.int32).reshape(-1, 2)], 255)
    return m

def mask_to_yolo_lines(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    H, W = mask.shape; lines = []
    for c in cnts:
        if cv2.contourArea(c) < 50: continue
        c = c.reshape(-1, 2).astype(float); c[:, 0] /= W; c[:, 1] /= H
        lines.append("0 " + " ".join(f"{v:.6f}" for v in c.flatten()))
    return lines

# Albumentations: Hue 변형 없음 (수박 색 피처 보존)
aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.1, rotate_limit=45, p=0.4),
    A.RandomBrightnessContrast(p=0.4),
    A.RandomShadow(p=0.2),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
])
N_AUG = 10   # 96 -> ~960장. 디스크 부족 시 5로 낮추기

## Phase 4. Fold 디렉터리 빌드 (val=원본, train=원본+증강)

In [ ]:
def build_fold(k, tr_idx, va_idx):
    root = f"/content/folds/fold{k}"
    for s in ["images/train", "images/val", "labels/train", "labels/val"]:
        os.makedirs(f"{root}/{s}", exist_ok=True)

    # Validation: 원본만
    for j in va_idx:
        im = imgs[valid_ids[j]]; fn = im["file_name"]; stem = os.path.splitext(fn)[0]
        cv2.imwrite(f"{root}/images/val/{fn}", cv2.imread(f"{ROOT}/{fn}"))
        write_yolo_label(body_by_img[im['id']], im, f"{root}/labels/val/{stem}.txt")

    # Train: 원본 + 증강
    for j in tr_idx:
        im = imgs[valid_ids[j]]; fn = im["file_name"]; stem = os.path.splitext(fn)[0]
        img = cv2.imread(f"{ROOT}/{fn}"); mask = poly_to_mask(body_by_img[im['id']], im)
        cv2.imwrite(f"{root}/images/train/{fn}", img)
        write_yolo_label(body_by_img[im['id']], im, f"{root}/labels/train/{stem}.txt")
        for n in range(N_AUG):
            o = aug(image=img, mask=mask)
            lines = mask_to_yolo_lines(o["mask"])
            if not lines: continue
            cv2.imwrite(f"{root}/images/train/{stem}_a{n}.jpg", o["image"])
            open(f"{root}/labels/train/{stem}_a{n}.txt", "w").write("\n".join(lines))

    open(f"{root}/data.yaml", "w").write(
        f"path: {root}\ntrain: images/train\nval: images/val\nnames:\n  0: body\n")
    return f"{root}/data.yaml"

print("디스크 여유 확인:")
!df -h /content | tail -1

## Phase 5. Fold별 학습 + 평가 (내부 증강 OFF)
⏱️ 5 fold × 최대 300 epoch → 무료 T4 기준 **수 시간~십수 시간**. 결과는 Drive에 저장됩니다.
세션 끊김이 걱정되면 루프를 `for k in [0]:` 처럼 **fold를 나눠** 실행하세요.

In [ ]:
from ultralytics import YOLO

SAVE = "/content/drive/MyDrive/watermelon/runs"
results = {}

for k, (tr, va) in enumerate(folds):
    print(f"\n===== Fold {k} 빌드 =====")
    yaml = build_fold(k, tr, va)

    model = YOLO("yolov8s-seg.pt")
    model.train(
        data=yaml, epochs=300, patience=50, batch=16, imgsz=640,
        optimizer="AdamW", lr0=0.001,
        # 내부 증강 전부 OFF (오프라인 증강만 사용)
        mosaic=0.0, mixup=0.0, degrees=0.0, translate=0.0, scale=0.0,
        shear=0.0, perspective=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
        fliplr=0.0, flipud=0.0,
        project=SAVE, name=f"fold{k}", exist_ok=True,
    )
    m = model.val(data=yaml)
    results[k] = {"map50": float(m.seg.map50), "map": float(m.seg.map)}
    print(f"[fold{k}] Mask mAP50={m.seg.map50:.4f}  mAP50-95={m.seg.map:.4f}")

print("\n결과:", results)

## Phase 6. 5-Fold 평균 = 최종 성능

In [ ]:
m50 = np.array([results[k]["map50"] for k in results])
m95 = np.array([results[k]["map"]   for k in results])
print("===== 최종 (5-Fold 평균) =====")
print(f"Mask mAP50    : {m50.mean():.4f}  (fold별 {np.round(m50, 3).tolist()})")
print(f"Mask mAP50-95 : {m95.mean():.4f}  (fold별 {np.round(m95, 3).tolist()})")

## Phase 7. (선택) 배포용 최종 모델 — 전체 데이터로 재학습
교차검증으로 성능을 확정했으면, 전체 120장(+증강)으로 1개 모델을 학습해 `best.pt`로 사용합니다.

In [ ]:
all_idx = list(range(len(valid_ids)))
yaml_full = build_fold("_full", all_idx, all_idx[:12])   # val은 형식상 일부
final = YOLO("yolov8s-seg.pt")
final.train(
    data=yaml_full, epochs=300, patience=50, batch=16, imgsz=640,
    optimizer="AdamW", lr0=0.001,
    mosaic=0.0, mixup=0.0, degrees=0.0, translate=0.0, scale=0.0,
    shear=0.0, perspective=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    fliplr=0.0, flipud=0.0,
    project="/content/drive/MyDrive/watermelon/runs", name="final", exist_ok=True,
)
print("최종 모델:", "/content/drive/MyDrive/watermelon/runs/final/weights/best.pt")

## Phase 8. 추론 테스트
학습된 모델로 수박 사진의 body 마스크를 확인합니다.

In [ ]:
import matplotlib.pyplot as plt

best = "/content/drive/MyDrive/watermelon/runs/final/weights/best.pt"
m = YOLO(best)
r = m.predict(f"{ROOT}/" + imgs[valid_ids[0]]["file_name"], conf=0.3)[0]
plt.figure(figsize=(8, 8))
plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off"); plt.show()